# LinkedIn Job Postings 2023–2024 — Preprocessing

My own preprocessing pipeline. Goal: turn 11 normalised CSVs into **one modelling table**
with an annualised salary target, ready for regression.

**Scope:** loading → target construction → aggregation → joining → cleaning → feature prep →
train/test split. No modelling here.

## Design decisions, up front

| Decision | Choice | Why |
|---|---|---|
| Salary from `min`/`max` | **midpoint** `(min+max)/2` | 83% of salary rows are a *range* with no median. Using `med_salary` alone throws away 5 of every 6 labels. |
| Pay period | annualise everything | `HOURLY 20` and `YEARLY 55000` are the same column; incomparable until normalised. |
| Join type — features | **left** onto postings | A job with no listed skills is still a valid job. Inner joins would silently delete it. |
| Join type — target | **inner** at the very end | A row with no salary cannot train a salary model. This is the *only* place dropping rows is correct. |
| Target transform | `log1p` | Salary is right-skewed; log makes errors proportional (a \$10k miss on \$50k ≠ on \$500k). |
| Missing `n_skills` | fill **0** | Structural zero, not missing data — "no skills listed" *is* the value. |
| Missing `experience_level` | keep NaN + **flag** | 23.7% missing and probably non-random. Imputing invents a level; a flag lets the model use "unstated" as signal. |

## Every one-to-many table is aggregated *before* joining
`job_skills` averages 1.69 rows per job. Merging it raw turns 123,849 postings into ~210,000
duplicated rows and silently corrupts every downstream statistic. Aggregate first, always.

---
## 0. Setup

In [1]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

SEED = 42
np.random.seed(SEED)

BASE = 'data'
OUT  = 'processed'
os.makedirs(OUT, exist_ok=True)

path = lambda *parts: os.path.join(BASE, *parts)


# Print shape after every major step so I can watch the data change.
def show(df, label):
    print(f'  {label:<52} {df.shape[0]:>8,} rows x {df.shape[1]:>3} cols')


print('Setup done.')

Setup done.


---
## 1. Load

`postings.csv` is 493 MB, and almost all of that is two free-text columns
(`description`, `skills_desc`). I load the structured columns first and handle
the text separately in a streaming pass, so the notebook never holds ~2 GB of
job descriptions in memory at once.

In [2]:
# Structured columns only. Deliberately excluded: description, skills_desc (huge text,
# handled in the next cell), and dead-end columns (urls, posting_domain, application_type).
POSTING_COLS = [
    'job_id', 'company_id', 'company_name', 'title', 'location',
    'formatted_work_type', 'work_type', 'formatted_experience_level',
    'remote_allowed', 'sponsored', 'views', 'applies',
    'listed_time', 'original_listed_time', 'expiry',
    # Salary columns: loaded ONLY to build the target and to validate my maths.
    # They are dropped from the feature matrix later -- they ARE the target.
    'min_salary', 'med_salary', 'max_salary', 'pay_period',
    'currency', 'compensation_type', 'normalized_salary',
]

postings = pd.read_csv(path('postings.csv'), usecols=POSTING_COLS, low_memory=False)

salaries       = pd.read_csv(path('jobs', 'salaries.csv'))
job_skills     = pd.read_csv(path('jobs', 'job_skills.csv'))
job_industries = pd.read_csv(path('jobs', 'job_industries.csv'))
benefits       = pd.read_csv(path('jobs', 'benefits.csv'))

companies          = pd.read_csv(path('companies', 'companies.csv'), low_memory=False)
company_industries = pd.read_csv(path('companies', 'company_industries.csv'))
company_spec       = pd.read_csv(path('companies', 'company_specialities.csv'))
employee_counts    = pd.read_csv(path('companies', 'employee_counts.csv'))

skills_map     = pd.read_csv(path('mappings', 'skills.csv'))
industries_map = pd.read_csv(path('mappings', 'industries.csv'))

print('LOADED')
for name, d in [('postings', postings), ('salaries', salaries), ('job_skills', job_skills),
                ('job_industries', job_industries), ('benefits', benefits),
                ('companies', companies), ('company_industries', company_industries),
                ('company_spec', company_spec), ('employee_counts', employee_counts),
                ('skills_map', skills_map), ('industries_map', industries_map)]:
    show(d, name)

# Sanity check the assumption the whole pipeline rests on: job_id is a true primary key.
assert postings['job_id'].is_unique, 'job_id is not unique -- every join below is unsafe'
print('\njob_id is unique in postings -- safe to use as the join spine.')

LOADED
  postings                                              123,849 rows x  22 cols
  salaries                                               40,785 rows x   8 cols
  job_skills                                            213,768 rows x   2 cols
  job_industries                                        164,808 rows x   2 cols
  benefits                                               67,943 rows x   3 cols
  companies                                              24,473 rows x  10 cols
  company_industries                                     24,375 rows x   2 cols
  company_spec                                          169,387 rows x   2 cols
  employee_counts                                        35,787 rows x   4 cols
  skills_map                                                 35 rows x   2 cols
  industries_map                                            422 rows x   2 cols

job_id is unique in postings -- safe to use as the join spine.


In [3]:
# Description length is a genuinely useful feature (longer posts skew senior / corporate),
# but the text itself is 400+ MB. Stream it in chunks, keep the two numbers, discard the text.
chunks = []
for chunk in pd.read_csv(path('postings.csv'), usecols=['job_id', 'description'],
                         chunksize=20_000, low_memory=False):
    text = chunk['description'].fillna('')
    chunks.append(pd.DataFrame({
        'job_id': chunk['job_id'],
        'desc_word_count': text.str.split().str.len(),
        'desc_char_count': text.str.len(),
    }))

desc_features = pd.concat(chunks, ignore_index=True)
del chunks

show(desc_features, 'desc_features (streamed, text discarded)')
print()
print(desc_features['desc_word_count'].describe().round(1).to_string())

  desc_features (streamed, text discarded)              123,849 rows x   3 cols

count    123849.0
mean        523.0
std         301.9
min           0.0
25%         298.0
50%         477.0
75%         696.0
max        3400.0


---
## 2. Build the salary target

Two problems have to be solved before salary is usable as a target.

**Problem 1 — the range/point split.** In `salaries.csv`, `min`/`max` and `med` are
*mutually exclusive*: 33,947 rows have a range and no median, 6,838 have a median and no
range. No row has both. So the rule is: **use `med_salary` when present, otherwise the
midpoint of `min` and `max`.** Taking only `med_salary` would silently discard 83% of the labels.

**Problem 2 — mixed pay periods.** `YEARLY 23,768 · HOURLY 16,289 · MONTHLY 539 ·
WEEKLY 180 · BIWEEKLY 9`. All must be annualised onto one scale.

Hourly uses **2080 = 40 h/week × 52 weeks**. This is a convention, and it's an assumption
worth stating: it treats every hourly role as full-time, so genuinely part-time hourly
roles get inflated to a full-time-equivalent figure.

In [4]:
sal = salaries.copy()

# --- Problem 1: one salary point per posting -------------------------------------------
# Prefer the stated median. Fall back to the midpoint of the range.
# Midpoint (not min, not max) because a posted range is an employer's negotiating band --
# its centre is the least biased single estimate of the eventual offer.
sal['salary_point'] = sal['med_salary'].fillna(
    sal[['min_salary', 'max_salary']].mean(axis=1)
)

# --- Problem 2: annualise ---------------------------------------------------------------
PERIOD_MULTIPLIER = {
    'HOURLY':   2080,   # 40 h/week x 52 weeks -- the standard FTE convention
    'DAILY':     260,   # 5 days x 52 weeks
    'WEEKLY':     52,
    'BIWEEKLY':   26,   # every two weeks -> 26 pay periods, NOT 24 (that's semi-monthly)
    'MONTHLY':    12,
    'YEARLY':      1,
}
multiplier = sal['pay_period'].str.upper().map(PERIOD_MULTIPLIER)

# Any pay_period I haven't mapped becomes NaN rather than silently defaulting to 1x --
# a wrong multiplier is far worse than a missing row.
unmapped = sal.loc[multiplier.isna(), 'pay_period'].value_counts()
if len(unmapped):
    print('WARNING -- unmapped pay periods:\n', unmapped.to_string())

sal['annual_salary'] = sal['salary_point'] * multiplier

print('Rows recovered by the midpoint rule:')
print(f'  med_salary only (what a naive version keeps) : '
      f'{(sal["med_salary"].notna() & multiplier.notna()).sum():>7,}')
print(f'  med OR midpoint of range (my rule)           : '
      f'{sal["annual_salary"].notna().sum():>7,}')
print(f'  -> {sal["annual_salary"].notna().sum() / (sal["med_salary"].notna()).sum():.1f}x more labels\n')

show(sal, 'salaries with annual_salary')
print()
print('Median annualised salary by pay period (sanity check -- these should be comparable):')
print(sal.groupby('pay_period')[['salary_point', 'annual_salary']].median().round(0).to_string())

Rows recovered by the midpoint rule:
  med_salary only (what a naive version keeps) :   6,838
  med OR midpoint of range (my rule)           :  40,785
  -> 6.0x more labels

  salaries with annual_salary                            40,785 rows x  10 cols

Median annualised salary by pay period (sanity check -- these should be comparable):
            salary_point  annual_salary
pay_period                             
BIWEEKLY         60492.0      1572805.0
HOURLY              25.0        52000.0
MONTHLY           4308.0        51702.0
WEEKLY            2099.0       109148.0
YEARLY          103000.0       103000.0


In [5]:
# --- Validate my annualisation against the dataset's own normalized_salary --------------
# postings.csv ships a precomputed `normalized_salary`. I did NOT use it to build my target,
# so comparing against it is an independent check that my midpoint + multiplier logic is right.
check = (sal[['job_id', 'annual_salary']]
         .merge(postings[['job_id', 'normalized_salary']], on='job_id', how='inner')
         .dropna())

exact = (check['annual_salary'].round(0) == check['normalized_salary'].round(0)).mean()
print(f'Rows compared      : {len(check):,}')
print(f'Exact match        : {exact:.2%}')
print(f'Correlation        : {check["annual_salary"].corr(check["normalized_salary"]):.6f}')
print()
if exact > 0.99:
    print('My independently-derived target reproduces the official normalized_salary exactly.')
    print('Confirms: midpoint-of-range x period multiplier is the dataset author\'s own definition.')
else:
    print('Mismatch -- investigate before trusting the target.')
    print(check[check['annual_salary'].round(0) != check['normalized_salary'].round(0)].head())

Rows compared      : 36,073
Exact match        : 100.00%


Correlation        : 1.000000

My independently-derived target reproduces the official normalized_salary exactly.
Confirms: midpoint-of-range x period multiplier is the dataset author's own definition.


In [6]:
# --- Currency ---------------------------------------------------------------------------
# 40,770 of 40,785 rows are USD. The 15 non-USD rows (EUR/CAD/GBP/AUD/BBD) are NOT converted
# anywhere in this dataset, so a "50000" in EUR would sit on the same axis as USD and be wrong.
# Dropping 15 rows costs nothing; an FX conversion would need rate data I don't have.
before = len(sal)
sal = sal[sal['currency'] == 'USD']
print(f'Non-USD rows dropped: {before - len(sal)}')
show(sal, 'after currency filter')

# --- Implausible values ------------------------------------------------------------------
# Bounds are a judgement call, so state the reasoning:
#   Floor $10,000  -- below US federal minimum wage FTE (7.25 x 2080 = $15,080). Anything
#                     under this is a data-entry error (e.g. an hourly rate typed in a
#                     yearly field), not a real job.
#   Ceiling $500,000 -- keeps genuine executive pay, cuts the handful of absurd rows
#                     (e.g. an annual figure typed into an hourly field -> $100M+).
# I clip rather than winsorise because these are errors, not extreme-but-real observations.
SALARY_FLOOR, SALARY_CEILING = 10_000, 500_000

before = len(sal)
sal = sal[sal['annual_salary'].between(SALARY_FLOOR, SALARY_CEILING)]
print(f'\nOutliers dropped: {before - len(sal):,} '
      f'({(before - len(sal)) / before:.2%} of USD salary rows)')
show(sal, 'after outlier filter')

# One clean row per job_id. salaries.csv is already unique per job, but grouping makes that
# guarantee explicit -- if the source ever changes, this fails loudly instead of fanning out.
salary_target = sal.groupby('job_id', as_index=False)['annual_salary'].median()
assert salary_target['job_id'].is_unique
show(salary_target, 'salary_target (1 row per job_id)')

print()
print(salary_target['annual_salary'].describe().round(0).to_string())

Non-USD rows dropped: 15
  after currency filter                                  40,770 rows x  10 cols

Outliers dropped: 542 (1.33% of USD salary rows)
  after outlier filter                                   40,228 rows x  10 cols
  salary_target (1 row per job_id)                       40,228 rows x   2 cols

count     40228.0
mean      96664.0
std       57418.0
min       10000.0
25%       52000.0
50%       83000.0
75%      125328.0
max      500000.0


---
## 3. Collapse the one-to-many tables

Every table below has **more than one row per key**. Each gets aggregated to exactly one
row per key *before* it goes anywhere near a join. This is the single most important
mechanical step in the whole notebook.

### Skills → 35 binary columns
`job_skills` is a long table (job_id, skill_abr). I pivot it wide so each of the 35 skill
categories becomes its own 0/1 column, plus a `n_skills` count.

Why one-hot rather than label-encoding the skill? Because skills are **not ordinal and not
exclusive** — a job can be both `IT` and `MGMT`. Encoding `IT=1, SALE=2` would tell a linear
model that SALE is "twice" IT, which is meaningless. With only 35 categories the width cost
is trivial.

In [7]:
# Long -> wide. aggfunc='max' (not 'sum') so a duplicated (job, skill) pair stays 0/1
# instead of becoming 2.
skills_wide = (job_skills
               .assign(present=1)
               .pivot_table(index='job_id', columns='skill_abr',
                            values='present', aggfunc='max', fill_value=0))

# n_skills computed here, while the frame is still nothing but skill flags.
skills_wide['n_skills'] = skills_wide.sum(axis=1)

skills_wide.columns = [c if c == 'n_skills' else f'skill_{c}' for c in skills_wide.columns]
skills_wide = skills_wide.reset_index()

show(job_skills,   'job_skills (long, 1.69 rows/job)')
show(skills_wide,  'skills_wide (1 row/job, 35 flags + count)')
print()
print(skills_wide.head(3).iloc[:, :10].to_string())
print(f'\nAverage skills per job: {skills_wide["n_skills"].mean():.2f}')

  job_skills (long, 1.69 rows/job)                      213,768 rows x   2 cols
  skills_wide (1 row/job, 35 flags + count)             126,807 rows x  37 cols

    job_id  skill_ACCT  skill_ADM  skill_ADVR  skill_ANLS  skill_ART  skill_BD  skill_CNSL  skill_CUST  skill_DIST
0   921716           0          0           0           0          0         0           0           0           0
1  1218575           0          0           0           0          0         0           0           0           0
2  1829192           0          0           0           0          0         0           0           0           0

Average skills per job: 1.69


In [8]:
# --- Benefits: count + flags for the most common types ----------------------------------
n_benefits = benefits.groupby('job_id').size().reset_index(name='n_benefits')

TOP_BENEFITS = benefits['type'].value_counts().head(8).index.tolist()
benefits_wide = (benefits[benefits['type'].isin(TOP_BENEFITS)]
                 .assign(present=1)
                 .pivot_table(index='job_id', columns='type',
                              values='present', aggfunc='max', fill_value=0))
benefits_wide.columns = ['benefit_' + c.lower().replace(' ', '_').replace('/', '_')
                         for c in benefits_wide.columns]
benefits_wide = (benefits_wide.reset_index()
                 .merge(n_benefits, on='job_id', how='outer'))

show(benefits,      'benefits (long, 2.26 rows/job)')
show(benefits_wide, 'benefits_wide (1 row/job)')
print('Top benefit types:', TOP_BENEFITS)

  benefits (long, 2.26 rows/job)                         67,943 rows x   3 cols
  benefits_wide (1 row/job)                              30,023 rows x  10 cols
Top benefit types: ['401(k)', 'Medical insurance', 'Vision insurance', 'Disability insurance', 'Dental insurance', 'Tuition assistance', 'Commuter benefits', 'Paid maternity leave']


In [9]:
# --- Job industries: count + one primary industry ---------------------------------------
# There is no "primary" flag in the data, so picking one is arbitrary. I sort by industry_id
# and take the first: arbitrary, but DETERMINISTIC -- rerunning gives the same answer.
# (Taking whatever row happens to come first in file order would make the pipeline
# non-reproducible if the source file is ever re-sorted.)
job_ind = job_industries.merge(industries_map, on='industry_id', how='left')

n_industries = job_ind.groupby('job_id').size().reset_index(name='n_industries')
primary_industry = (job_ind.sort_values(['job_id', 'industry_id'])
                    .drop_duplicates('job_id', keep='first')
                    [['job_id', 'industry_name']]
                    .rename(columns={'industry_name': 'job_industry'}))

job_ind_agg = n_industries.merge(primary_industry, on='job_id', how='left')

show(job_industries, 'job_industries (long, 1.30 rows/job)')
show(job_ind_agg,    'job_ind_agg (1 row/job)')
print()
print(job_ind_agg['job_industry'].value_counts().head(8).to_string())

  job_industries (long, 1.30 rows/job)                  164,808 rows x   2 cols
  job_ind_agg (1 row/job)                               127,125 rows x   3 cols

job_industry
Hospitals and Health Care        17214
Retail                            9381
Staffing and Recruiting           8569
IT Services and IT Consulting     7479
Financial Services                6027
Software Development              4992
Construction                      3128
Manufacturing                     3025


In [10]:
# --- Employee counts: LATEST snapshot per company ----------------------------------------
# This table is a time series: 35,787 rows over 24,473 companies (1.46 snapshots each,
# 3,531 distinct timestamps). Joining it raw would duplicate postings for every company
# that was crawled twice. Sorting by time and keeping the last gives one current snapshot.
company_metrics = (employee_counts
                   .sort_values('time_recorded')
                   .drop_duplicates('company_id', keep='last')
                   [['company_id', 'employee_count', 'follower_count']])

assert company_metrics['company_id'].is_unique
show(employee_counts,  'employee_counts (1.46 snapshots/company)')
show(company_metrics,  'company_metrics (1 row/company, latest)')

# --- Company industry: dedupe (looks 1:1, but has 10 duplicate company_ids) --------------
company_ind = (company_industries
               .sort_values(['company_id', 'industry'])
               .drop_duplicates('company_id', keep='first')
               .rename(columns={'industry': 'company_industry'}))
show(company_industries, 'company_industries (raw, 10 dupes)')
show(company_ind,        'company_ind (deduped)')

# --- Specialities: count only. The text values are far too high-cardinality to encode. ----
n_spec = company_spec.groupby('company_id').size().reset_index(name='n_specialities')
show(n_spec, 'n_specialities (1 row/company)')

# --- Company profile: assemble everything company-level into ONE frame -------------------
company_profile = (companies[['company_id', 'company_size', 'state', 'country', 'city']]
                   .rename(columns={'state': 'company_state',
                                    'country': 'company_country',
                                    'city': 'company_city'})
                   .merge(company_metrics, on='company_id', how='left')
                   .merge(company_ind,     on='company_id', how='left')
                   .merge(n_spec,          on='company_id', how='left'))

assert company_profile['company_id'].is_unique
show(company_profile, 'company_profile (1 row/company)')

  employee_counts (1.46 snapshots/company)               35,787 rows x   4 cols
  company_metrics (1 row/company, latest)                24,473 rows x   3 cols
  company_industries (raw, 10 dupes)                     24,375 rows x   2 cols
  company_ind (deduped)                                  24,365 rows x   2 cols
  n_specialities (1 row/company)                         17,780 rows x   2 cols
  company_profile (1 row/company)                        24,473 rows x   9 cols


---
## 4. Join

`postings` is the **spine**. Everything else is `left`-joined onto it.

**Why left and not inner?** An inner join answers "jobs that have skills AND benefits AND a
matching company" — that's a different, much smaller, and self-selecting population. A job with
no listed benefits is still a real job; deleting it biases the sample toward well-documented
postings. Left join keeps the spine intact and represents absence honestly as NaN, which I then
decide how to handle *explicitly* in the next section.

The row count after every left join must stay **exactly 123,849**. If it grows, an aggregation
above failed and the data has silently fanned out. That's what the assert is for.

In [11]:
df = postings.copy()
show(df, 'START: postings spine')

for name, right, key in [
    ('desc_features',   desc_features,   'job_id'),
    ('skills_wide',     skills_wide,     'job_id'),
    ('benefits_wide',   benefits_wide,   'job_id'),
    ('job_ind_agg',     job_ind_agg,     'job_id'),
    ('company_profile', company_profile, 'company_id'),
]:
    rows_before = len(df)
    df = df.merge(right, on=key, how='left')
    # A left join onto a unique-keyed right table cannot change the row count.
    # If it does, the right table was not properly aggregated -- fail loudly, don't limp on.
    assert len(df) == rows_before, (
        f'{name} fanned out: {rows_before:,} -> {len(df):,}. Right table key is not unique.')
    show(df, f'+ {name}')

print(f'\nSpine intact: {len(df):,} rows (expected 123,849)')

  START: postings spine                                 123,849 rows x  22 cols
  + desc_features                                       123,849 rows x  24 cols


  + skills_wide                                         123,849 rows x  60 cols


  + benefits_wide                                       123,849 rows x  69 cols


  + job_ind_agg                                         123,849 rows x  71 cols


  + company_profile                                     123,849 rows x  79 cols

Spine intact: 123,849 rows (expected 123,849)


---
## 5. Clean

Three separate problems that get confused with each other constantly:

1. **Leakage / unavailable-at-prediction-time** — `views` and `applies` only exist *after* a
   posting goes live. A model that uses them cannot score a brand-new posting, and it's
   learning popularity, not pay. Drop.
2. **Zero variance** — `sponsored` is `0` for all 123,849 rows; `compensation_type` is
   `BASE_SALARY` for all of them. A constant column carries no information. Drop.
3. **Structural zeros vs genuine missingness** — this is the important one.

`remote_allowed` is the trap: its **only** observed value is `1.0`, with 108,603 NaN. Those NaN
are not unknown — they mean *not remote*. Treating them as missing and dropping the column (88%
null!) would throw away a strong, complete feature. Same logic for `n_skills` / `n_benefits`:
NaN came from the left join and means "none listed", which is the number zero.

These fills are **deterministic** — they use no information from the data distribution — so
doing them before the train/test split is safe. A mean or median fill is *not* deterministic
and must happen inside a pipeline, after the split. That distinction is the whole game.

In [12]:
# --- 1 & 2: drop leakage, zero-variance, and target-derived columns ----------------------
LEAKY = ['views', 'applies']                       # post-publication engagement
CONSTANT = ['sponsored', 'compensation_type']      # single value across all rows
TARGET_DERIVED = ['min_salary', 'med_salary', 'max_salary', 'pay_period',
                  'currency', 'normalized_salary'] # these ARE the target
IDENTIFIERS = ['company_name']                     # 24k-cardinality string, company_id covers it

for label, cols in [('leaky', LEAKY), ('constant', CONSTANT),
                    ('target-derived', TARGET_DERIVED), ('identifier', IDENTIFIERS)]:
    drop = [c for c in cols if c in df.columns]
    df = df.drop(columns=drop)
    print(f'  dropped {label:<15}: {drop}')

show(df, 'after dropping unusable columns')

  dropped leaky          : ['views', 'applies']
  dropped constant       : ['sponsored', 'compensation_type']


  dropped target-derived : ['min_salary', 'med_salary', 'max_salary', 'pay_period', 'currency', 'normalized_salary']
  dropped identifier     : ['company_name']
  after dropping unusable columns                       123,849 rows x  68 cols


In [13]:
# --- 3: structural zeros --------------------------------------------------------------
# NaN here means "absent", which is a real, known value -- not missing information.
df['is_remote'] = df['remote_allowed'].fillna(0).astype(int)
df = df.drop(columns=['remote_allowed'])

COUNT_COLS = (['n_skills', 'n_benefits', 'n_industries', 'n_specialities']
              + [c for c in df.columns if c.startswith(('skill_', 'benefit_'))])
df[COUNT_COLS] = df[COUNT_COLS].fillna(0)

print(f'is_remote: {df["is_remote"].sum():,} remote / {len(df):,} total '
      f'({df["is_remote"].mean():.1%})')
print(f'Structural zeros filled in {len(COUNT_COLS)} count/flag columns.')
show(df, 'after structural-zero fills')

# --- What is still genuinely missing? --------------------------------------------------
# Everything below is real missingness and is deliberately LEFT as NaN, to be imputed inside
# a pipeline after the split (or handled natively by a gradient-boosting model).
missing = (df.isna().mean() * 100).round(1)
missing = missing[missing > 0].sort_values(ascending=False)
print('\nRemaining genuine missingness (left as NaN on purpose):')
print(missing.to_string())

is_remote: 15,246 remote / 123,849 total (12.3%)
Structural zeros filled in 47 count/flag columns.
  after structural-zero fills                           123,849 rows x  68 cols



Remaining genuine missingness (left as NaN on purpose):
formatted_experience_level    23.7
company_size                   5.4
company_industry               1.5
company_id                     1.4
company_country                1.4
company_state                  1.4
employee_count                 1.4
company_city                   1.4
follower_count                 1.4
job_industry                   1.2


---
## 6. Feature preparation

**Experience level** is ordinal — Internship < Entry < Associate < Mid-Senior < Director <
Executive — so it gets an integer map, not one-hot. Ordering is real information and one-hot
would throw it away.

But 23.7% of postings have no experience level. I do **not** impute it. Missingness here is
almost certainly not random (junior/high-churn roles skip the field more often), so a median
fill would both invent data and destroy that signal. Instead: keep the NaN and add an explicit
`exp_level_missing` flag, letting the model decide whether "unstated" is predictive.

**Dates.** I checked the range: the entire dataset is `2024-03-24` → `2024-04-20`. So
`post_month` and `post_quarter` are effectively constant and are useless features — a
copy-paste date-feature block would add two dead columns here. Only `day of week` has real
variance, so that's all I build.

In [14]:
# --- Ordinal: experience level -----------------------------------------------------------
EXPERIENCE_ORDER = {
    'Internship': 0, 'Entry level': 1, 'Associate': 2,
    'Mid-Senior level': 3, 'Director': 4, 'Executive': 5,
}
df['exp_level_num'] = df['formatted_experience_level'].map(EXPERIENCE_ORDER)
df['exp_level_missing'] = df['formatted_experience_level'].isna().astype(int)

print('Experience level:')
print(f'  mapped  : {df["exp_level_num"].notna().sum():,}')
print(f'  missing : {df["exp_level_missing"].sum():,} ({df["exp_level_missing"].mean():.1%}) '
      f'-- kept as NaN + flag, NOT imputed')

# --- company_size is ALREADY numeric (1-7) in this dataset --------------------------------
# Worth checking rather than assuming: an ordinal-letter map (A..I) would silently produce
# an all-NaN column here, because these values are floats.
print(f'\ncompany_size dtype: {df["company_size"].dtype}, '
      f'values: {sorted(df["company_size"].dropna().unique())}')
df['company_size_num'] = df['company_size']  # already ordinal-encoded at source

Experience level:
  mapped  : 94,440
  missing : 29,409 (23.7%) -- kept as NaN + flag, NOT imputed

company_size dtype: float64, values: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0), np.float64(6.0), np.float64(7.0)]


In [15]:
# --- Dates ---------------------------------------------------------------------------------
listed_dt = pd.to_datetime(df['listed_time'], unit='ms', errors='coerce')
expiry_dt = pd.to_datetime(df['expiry'],      unit='ms', errors='coerce')

print(f'Posting window: {listed_dt.min():%Y-%m-%d} to {listed_dt.max():%Y-%m-%d}')
print(f'Distinct months: {listed_dt.dt.month.nunique()} -> month/quarter features would be '
      f'constant, so they are NOT built.')

df['post_dayofweek'] = listed_dt.dt.dayofweek          # real variance: Thu/Fri dominate
df['post_is_weekend'] = (listed_dt.dt.dayofweek >= 5).astype(int)
df['listing_duration_days'] = (expiry_dt - listed_dt).dt.total_seconds() / 86_400

df = df.drop(columns=['listed_time', 'original_listed_time', 'expiry'])
show(df, 'after date features')

Posting window: 2024-03-24 to 2024-04-20
Distinct months: 2 -> month/quarter features would be constant, so they are NOT built.
  after date features                                   123,849 rows x  71 cols


In [16]:
# --- Location: extract state from the free-text "City, ST" field --------------------------
# 8,526 distinct location strings is too high-cardinality to encode directly, but the state
# suffix is both low-cardinality and where most of the geographic pay variation lives.
loc = df['location'].fillna('')
state = loc.str.rsplit(',', n=1).str[-1].str.strip()
# Keep only real 2-letter codes; everything else ("United States", "Anywhere") -> NaN.
df['job_state'] = state.where(state.str.fullmatch(r'[A-Z]{2}'), np.nan)

print(f'State parsed for {df["job_state"].notna().mean():.1%} of postings')
print(df['job_state'].value_counts().head(8).to_string())

# --- Title: cheap text signal without full NLP --------------------------------------------
title = df['title'].fillna('').str.lower()
df['title_word_count'] = title.str.split().str.len()
for kw in ['senior', 'junior', 'manager', 'director', 'engineer', 'analyst', 'intern']:
    df[f'title_has_{kw}'] = title.str.contains(kw, regex=False).astype(int)

show(df, 'after location + title features')

State parsed for 84.9% of postings
job_state
CA    11484
TX    10271
NY     6044
FL     5907
NC     4927
IL     4480
PA     4133
VA     3660


  after location + title features                       123,849 rows x  80 cols


In [17]:
# --- Skew: log-transform the heavy-tailed company counts ----------------------------------
# employee_count and follower_count span ~1 to ~10 million. Untransformed, distance-based and
# linear models are dominated entirely by a handful of mega-corps. log1p (not log) because
# zero is a legitimate value and log(0) is undefined.
for col in ['employee_count', 'follower_count']:
    print(f'{col:<16} raw skew {df[col].skew():>10.1f}  ->  log1p skew {np.log1p(df[col]).skew():>6.2f}')
    df[f'log_{col}'] = np.log1p(df[col])

show(df, 'after log transforms')

employee_count   raw skew        8.3  ->  log1p skew  -0.26
follower_count   raw skew       10.3  ->  log1p skew  -0.48
  after log transforms                                  123,849 rows x  82 cols


---
## 7. Attach the target and split

The target join is the **one place an inner join is correct**: a row with no salary cannot
train a salary model. Everything up to here was built on all 123,849 postings so the feature
logic is identical for labelled and unlabelled rows — which means the unlabelled 71% is still
available later for semi-supervised work or for predicting on unseen postings.

**Why `log1p` on the target?** Salary is strongly right-skewed. In raw dollars, squared error
is dominated by high earners and the model optimises for them. In log space, an error is
*proportional* — being \$10k off on a \$50k job is penalised like being \$100k off on a \$500k
job, which matches how anyone actually judges a salary estimate. Predictions come back with
`np.expm1()`.

**Split before any data-dependent transform.** Nothing below fits a scaler or an imputer —
those belong inside a `Pipeline` fitted on the training fold only. Fitting them here, on all
the data, would leak test-set statistics into training and inflate the score.

In [18]:
# INNER join -- deliberately drops the 71% of postings with no salary label.
rows_before = len(df)
model_df = df.merge(salary_target, on='job_id', how='inner')

print(f'Postings with features : {rows_before:,}')
print(f'Postings with a salary : {len(model_df):,} ({len(model_df) / rows_before:.1%})')
print(f'Dropped (unlabelled)   : {rows_before - len(model_df):,}')
show(model_df, 'model_df (labelled rows only)')

# Target: log space for training, raw kept for interpretable reporting.
model_df['log_salary'] = np.log1p(model_df['annual_salary'])
print()
print(f'Raw salary  -- skew {model_df["annual_salary"].skew():.2f}, '
      f'median ${model_df["annual_salary"].median():,.0f}')
print(f'Log salary  -- skew {model_df["log_salary"].skew():.2f}   <-- much closer to symmetric')

Postings with features : 123,849
Postings with a salary : 35,546 (28.7%)
Dropped (unlabelled)   : 88,303
  model_df (labelled rows only)                          35,546 rows x  83 cols

Raw salary  -- skew 1.67, median $82,500
Log salary  -- skew 0.06   <-- much closer to symmetric


In [19]:
# --- Assemble the final feature matrix ---------------------------------------------------
DROP_FROM_X = [
    'job_id',                                  # identifier
    'annual_salary', 'log_salary',             # targets
    'title', 'location',                       # raw text, superseded by engineered versions
    'formatted_experience_level',              # superseded by exp_level_num + flag
    'company_size',                            # superseded by company_size_num
    'work_type',                               # exact duplicate of formatted_work_type
    'employee_count', 'follower_count',        # superseded by log_ versions
    'company_city',                            # ~10k cardinality, company_state is enough
]

X = model_df.drop(columns=[c for c in DROP_FROM_X if c in model_df.columns])
y = model_df['log_salary']

categorical = X.select_dtypes(include='object').columns.tolist()
numeric = X.select_dtypes(include=np.number).columns.tolist()

print(f'Features: {X.shape[1]}  ({len(numeric)} numeric, {len(categorical)} categorical)')
print(f'\nCategorical (need encoding downstream): {categorical}')
print(f'  cardinality: {{ {", ".join(f"{c}: {X[c].nunique()}" for c in categorical)} }}')
show(X, 'X (feature matrix)')

# --- Split -------------------------------------------------------------------------------
# No stratification: this is regression. If the salary distribution were badly imbalanced I
# would stratify on quantile bins, but log_salary is near-symmetric so a random split is fine.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)

show(X_train, 'X_train')
show(X_test,  'X_test')
print(f'\nTrain median salary: ${np.expm1(y_train.median()):,.0f}')
print(f'Test  median salary: ${np.expm1(y_test.median()):,.0f}   <-- should be close')

Features: 73  (67 numeric, 6 categorical)

Categorical (need encoding downstream): ['formatted_work_type', 'job_industry', 'company_state', 'company_country', 'company_industry', 'job_state']


  cardinality: { formatted_work_type: 7, job_industry: 322, company_state: 416, company_country: 44, company_industry: 139, job_state: 51 }
  X (feature matrix)                                     35,546 rows x  73 cols
  X_train                                                28,436 rows x  73 cols
  X_test                                                  7,110 rows x  73 cols

Train median salary: $82,500
Test  median salary: $82,500   <-- should be close


In [20]:
# --- Persist -----------------------------------------------------------------------------
model_df.to_csv(os.path.join(OUT, 'model_df.csv'), index=False)
X_train.assign(log_salary=y_train).to_csv(os.path.join(OUT, 'train.csv'), index=False)
X_test.assign(log_salary=y_test).to_csv(os.path.join(OUT, 'test.csv'), index=False)

print(f'Saved to {OUT}:')
print(f'  model_df.csv  {model_df.shape[0]:>7,} x {model_df.shape[1]:>3}')
print(f'  train.csv     {X_train.shape[0]:>7,} x {X_train.shape[1] + 1:>3}')
print(f'  test.csv      {X_test.shape[0]:>7,} x {X_test.shape[1] + 1:>3}')

print('\n' + '=' * 74)
print('PIPELINE SUMMARY'.center(74))
print('=' * 74)
print(f'  123,849 postings loaded')
print(f'  -> {len(salary_target):,} with a usable annualised salary (midpoint rule)')
print(f'  -> {len(model_df):,} rows x {X.shape[1]} features after joins and cleaning')
print(f'  -> {len(X_train):,} train / {len(X_test):,} test')
print('=' * 74)

Saved to D:\Job_analytics\processed:
  model_df.csv   35,546 x  84
  train.csv      28,436 x  74
  test.csv        7,110 x  74

                             PIPELINE SUMMARY                             
  123,849 postings loaded
  -> 40,228 with a usable annualised salary (midpoint rule)
  -> 35,546 rows x 73 features after joins and cleaning
  -> 28,436 train / 7,110 test


---
## Handoff to modelling

Deliberately **not** done here, because it must be fitted on the training fold only:

- **Imputation** of `exp_level_num`, `company_size_num`, `log_employee_count`,
  `listing_duration_days` → `SimpleImputer` inside a `Pipeline`.
- **Scaling** → only needed for linear/SVM/KNN; trees don't care.
- **Categorical encoding** of `formatted_work_type` (7), `job_state` (~50),
  `job_industry` (~150), `company_industry` (~150), `company_country`.
  One-hot for the small ones; target encoding for industry needs out-of-fold fitting or it
  leaks the target directly.

Doing any of these here, before the split, would fit them on test data and inflate the score.

## Known limitations of this dataset

- **Selection bias.** Only 29% of postings disclose salary, and disclosure correlates with
  pay-transparency laws (CA, CO, NY, WA) and with larger employers. The model describes
  *salary-disclosing* postings, not the labour market.
- **One-month window.** 2024-03-24 → 2024-04-20. No seasonality can be learned.
- **Hourly → FTE assumption.** ×2080 treats every hourly role as full-time; genuinely
  part-time roles are inflated.
- **`skill_abr` is coarse.** 35 broad categories ("IT", "Engineering"), not actual
  technologies. Real skill extraction would need NLP over `description`.

- **`company_state` is dirty — do not use it as-is.** 416 distinct values, mixing 2-letter
  codes (`NY`), full names (`Texas`), and `0` used as a null placeholder. `job_state` (51
  values, regex-validated to `[A-Z]{2}`) is the clean geography feature; `company_state`
  needs normalising before it is encoded.
- **The BIWEEKLY sanity check earned its keep.** Median annualised BIWEEKLY pay came out at
  \$1.57M — those 9 rows hold *annual* figures mislabelled as biweekly. The \$500k ceiling
  removed them, but this is exactly why the per-period median is printed rather than assumed.
